# GEO Skin Aging Gene Expression Analysis

Reusable Python notebook for examining associations between age and gene expression in healthy human skin using [GSE18876](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE18876) (GPL5175).

## 1. Configuration

Change `GENE_ID` to an NCBI Gene ID. `YOUNG_MAX` is exclusive and `OLD_MIN` is inclusive. Run all cells from the top after changing settings.

In [ ]:
GSE_ID = "GSE18876"       # 健常ヒト皮膚の自然老化
GENE_ID = 327             # 例：327 = APEH。別のNCBI Gene IDに変更
YOUNG_MAX = 30            # 30歳未満
OLD_MIN = 60              # 60歳以上
DATA_DIR = "data"
RESULTS_DIR = "results"


## 2. Load GEO data and sample ages

Download the GEO Series and assign samples to Young, Middle, or Old age groups.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import GEOparse
from scipy.stats import spearmanr, ttest_ind
import statsmodels.api as sm

Path(DATA_DIR).mkdir(parents=True, exist_ok=True)
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

gse = GEOparse.get_GEO(geo=GSE_ID, destdir=DATA_DIR, silent=True)
print("サンプル数:", len(gse.gsms), "プラットフォーム:", list(gse.gpls))

records = []
for gsm_id, gsm in gse.gsms.items():
    age = np.nan
    for item in gsm.metadata.get("characteristics_ch1", []):
        match = re.search(r"^age:\s*(\d+(?:\.\d+)?)", item, re.I)
        if match:
            age = float(match.group(1))
            break
    records.append({"gsm_id": gsm_id, "age": age})
metadata = pd.DataFrame(records)
if metadata.age.isna().any():
    raise ValueError(f"年齢不明サンプル: {metadata.age.isna().sum()}件。メタデータの形式を確認してください。")
metadata["age_group"] = np.select(
    [metadata.age < YOUNG_MAX, metadata.age >= OLD_MIN],
    ["Young", "Old"], default="Middle"
)
print(metadata.age.describe())
print(metadata.age_group.value_counts())


## 3. Map NCBI Gene ID to platform probes

Match the exact NCBI Gene ID in GPL5175 `gene_assignment` records. Multiple matching probe sets are analyzed separately.

In [ ]:
def matching_gene_ids(assignment, gene_id):
    if pd.isna(assignment):
        return False
    for record in str(assignment).split("///"):
        parts = [part.strip() for part in record.split("//")]
        if len(parts) >= 5 and parts[4] == str(gene_id):
            return True
    return False

platform_ids = list(gse.gpls)
if len(platform_ids) != 1:
    raise ValueError("複数プラットフォームが存在します。GSMとGPLの対応を個別に処理してください。")
gpl = gse.gpls[platform_ids[0]]
annotation = gpl.table
required = {"ID", "gene_assignment"}
if not required.issubset(annotation.columns):
    raise ValueError("このGPLの注釈列が想定と異なります。")

candidates = annotation.loc[
    annotation["gene_assignment"].map(lambda v: matching_gene_ids(v, GENE_ID))
].copy()
if candidates.empty:
    raise ValueError(f"Gene ID {GENE_ID} の対応プローブセットが見つかりません。")

candidates["probe_id"] = candidates["ID"].astype(str)
print("対応プローブセット数:", len(candidates))
display(candidates[["probe_id", "gene_assignment", "category"]].head(20))


## 4. Extract expression and run statistical tests

For each probe set: OLS regression of expression on age, Spearman correlation, and Welch’s t-test (Old − Young). The table reports 95% confidence intervals, nominal p-values, and fold change calculated as `2 ** (Old mean − Young mean)` under the log2-expression assumption.

In [ ]:
# 各サンプルの発現表をID_REFでインデックス化（1回だけ）
indexed = {}
for gsm_id, gsm in gse.gsms.items():
    table = gsm.table[["ID_REF", "VALUE"]].copy()
    table["ID_REF"] = table["ID_REF"].astype(str)
    if table["ID_REF"].duplicated().any():
        raise ValueError(f"{gsm_id}: ID_REFが重複しています。")
    indexed[gsm_id] = table.set_index("ID_REF")["VALUE"]

summary_rows = []
all_data = []

for probe_id in candidates["probe_id"]:
    values = []
    for gsm_id, series in indexed.items():
        if probe_id not in series.index:
            raise ValueError(f"{gsm_id} にプローブセット {probe_id} がありません。")
        values.append({"gsm_id": gsm_id, "expression": float(series.loc[probe_id])})
    df = metadata.merge(pd.DataFrame(values), on="gsm_id", validate="one_to_one")
    if len(df) != len(metadata) or not np.isfinite(df.expression).all():
        raise ValueError(f"{probe_id}: 発現値に欠損・非有限値があります。")
    df["probe_id"] = probe_id
    df["gene_id"] = GENE_ID
    all_data.append(df)

    X = sm.add_constant(df["age"])
    model = sm.OLS(df["expression"], X).fit()
    rho, p_rho = spearmanr(df["age"], df["expression"])
    young = df.loc[df.age_group == "Young", "expression"]
    old = df.loc[df.age_group == "Old", "expression"]
    if len(young) < 2 or len(old) < 2:
        raise ValueError("若年群または高齢群の人数が2名未満です。閾値を見直してください。")
    welch = ttest_ind(old, young, equal_var=False)
    ci = welch.confidence_interval(confidence_level=0.95)
    diff = old.mean() - young.mean()
    p = welch.pvalue
    significance = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
    summary_rows.append({
        "gene_id": GENE_ID, "probe_id": probe_id, "n": len(df),
        "mean_expression": df.expression.mean(),
        "age_slope": model.params["age"],
        "age_slope_ci_low": model.conf_int().loc["age", 0],
        "age_slope_ci_high": model.conf_int().loc["age", 1],
        "age_p": model.pvalues["age"],
        "spearman_rho": rho, "spearman_p": p_rho,
        "young_n": len(young), "old_n": len(old),
        "young_mean": young.mean(), "old_mean": old.mean(),
        "old_minus_young_log2": diff,
        "difference_ci_low": ci.low, "difference_ci_high": ci.high,
        "fold_change_old_vs_young": 2 ** diff,
        "fold_change_ci_low": 2 ** ci.low,
        "fold_change_ci_high": 2 ** ci.high,
        "welch_p": welch.pvalue,
        "significance": significance,
        "significant_at_0_05": bool(p < 0.05),
    })

summary = pd.DataFrame(summary_rows)
all_expression = pd.concat(all_data, ignore_index=True)
display(summary)


## 5. Export tables and plots

Save `summary.csv`, `expression.csv`, age-regression plots, and group-comparison plots under `results/<GSE_ID>_GeneID_<GENE_ID>/`.

In [ ]:
# 結果の保存先
outdir = Path(RESULTS_DIR) / f"{GSE_ID}_GeneID_{GENE_ID}"
outdir.mkdir(parents=True, exist_ok=True)

summary.to_csv(
    outdir / "summary.csv",
    index=False,
    encoding="utf-8-sig"
)

all_expression.to_csv(
    outdir / "expression.csv",
    index=False,
    encoding="utf-8-sig"
)

for probe_id, df in all_expression.groupby("probe_id", sort=False):

    # ==========================================
    # 1. 年齢と発現量：線形回帰分析
    # ==========================================

    x = df["age"].to_numpy(dtype=float)
    y = df["expression"].to_numpy(dtype=float)

    model = sm.OLS(y, sm.add_constant(x)).fit()

    grid = np.linspace(x.min(), x.max(), 100)

    pred = model.get_prediction(
        sm.add_constant(grid)
    ).summary_frame(alpha=0.05)

    fig, ax = plt.subplots(figsize=(8, 5))

    ax.scatter(x, y, alpha=0.7)

    ax.plot(
        grid,
        pred["mean"],
        linewidth=2
    )

    ax.fill_between(
        grid,
        pred["mean_ci_lower"].to_numpy(),
        pred["mean_ci_upper"].to_numpy(),
        alpha=0.2
    )

    ax.set(
        title=f"{GSE_ID} | Gene ID {GENE_ID} | Probe {probe_id}",
        xlabel="Age (years)",
        ylabel="RMA expression (typically log2)"
    )

    fig.tight_layout()

    fig.savefig(
        outdir / f"probe_{probe_id}_age.png",
        dpi=200
    )

    plt.show()
    plt.close(fig)

    # ==========================================
    # 2. 若年群・高齢群の比較
    # ==========================================

    young = df.loc[
        df["age_group"] == "Young", "expression"
    ].dropna()

    old = df.loc[
        df["age_group"] == "Old", "expression"
    ].dropna()

    if len(young) < 2 or len(old) < 2:
        print(f"Probe {probe_id}: サンプル不足のため群間比較をスキップ")
        continue

    # Welchのt検定
    # Use the corresponding probe-level test result from summary
    probe_result = summary.loc[summary["probe_id"].astype(str) == str(probe_id)].iloc[0]
    p = float(probe_result["welch_p"])
    significance = probe_result["significance"]

    # ==========================================
    # 3. 箱ひげ図と有意差表示
    # ==========================================

    fig, ax = plt.subplots(figsize=(6, 5))

    ax.boxplot(
        [young, old],
        tick_labels=[
            f"Young (n={len(young)})",
            f"Old (n={len(old)})"
        ]
    )

    # 個々のサンプルを表示
    rng = np.random.default_rng(42)

    for i, group in enumerate([young, old], start=1):
        ax.scatter(
            i + rng.uniform(-0.09, 0.09, len(group)),
            group,
            alpha=0.55,
            s=20
        )

    ax.set(
        title=f"Gene ID {GENE_ID} | Probe {probe_id}",
        ylabel="RMA expression (typically log2)"
    )

    # p値と有意差判定を表示
    ax.text(
        0.5,
        0.98,
        f"Welch p = {p:.4g} ({significance})",
        transform=ax.transAxes,
        ha="center",
        va="top",
        fontsize=11,
        bbox=dict(
            facecolor="white",
            alpha=0.85,
            edgecolor="none"
        )
    )

    fig.tight_layout()

    fig.savefig(
        outdir / f"probe_{probe_id}_groups.png",
        dpi=200
    )

    plt.show()
    plt.close(fig)

print("出力先:", outdir.resolve())


## Notes and limitations

- The workflow is designed for GSE18876 / GPL5175 and is **not** a general-purpose parser for every GEO platform.
- Expression values are RMA-processed microarray intensities (typically log2 scale), not protein abundance or enzyme activity.
- Welch p-values are **unadjusted**. For many genes or probes, consider multiple-testing correction (e.g., Benjamini–Hochberg FDR).
- Changing age cutoffs after inspecting results is exploratory. Observational associations do not establish causality.
- Probe annotations can be outdated; confirm the mapping before biological interpretation.